# Farmspot crop condition classifier

Attach farmspot_quality_4crop_kaggle.zip and farmspot_kaggle_offline_dependencies.zip as Kaggle datasets. Enable a GPU accelerator and run all cells. The installation uses local package files.

The archive includes the pretrained tomato classifier and cropped images for tomato, cabbage, chili, and lettuce. Lettuce has a small exploratory holdout. The labels describe visible condition. See README.txt for source credits.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path
from zipfile import ZipFile

os.environ['YOLO_AUTOINSTALL'] = 'false'
inputs = Path('/kaggle/input')
archives = list(inputs.rglob('farmspot_kaggle_offline_dependencies.zip'))
if archives:
    wheel_dir = Path('/kaggle/working/farmspot_offline_wheels')
    wheel_dir.mkdir(parents=True, exist_ok=True)
    with ZipFile(archives[0]) as archive:
        archive.extractall(wheel_dir)
else:
    wheel = next(inputs.rglob('ultralytics-8.4.106-*.whl'), None)
    wheel_dir = wheel.parent if wheel else None

if wheel_dir is None:
    raise FileNotFoundError('Attach the Farmspot Offline Dependencies dataset using Add Input.')

subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '--no-index', '--disable-pip-version-check',
    '--find-links', str(wheel_dir), 'ultralytics==8.4.106',
])
import ultralytics
print('Offline setup complete. Ultralytics', ultralytics.__version__)

In [ ]:
import json
from pathlib import Path
from zipfile import ZipFile

import torch
from ultralytics import YOLO

inputs = Path('/kaggle/input')
working = Path('/kaggle/working')
archives = list(inputs.rglob('farmspot_quality_4crop_kaggle.zip'))
if archives:
    bundle = working / 'farmspot_quality_4crop_input'
    bundle.mkdir(exist_ok=True)
    with ZipFile(archives[0]) as archive:
        archive.extractall(bundle)
else:
    manifests = list(inputs.rglob('farmspot_quality_4crop_manifest.json'))
    if len(manifests) != 1:
        raise FileNotFoundError('Attach the Farmspot quality dataset as a Kaggle input.')
    bundle = manifests[0].parent

dataset = bundle / 'dataset'
weights = bundle / 'ripeness_yolo26s-cls_best.pt'
manifest = json.loads((bundle / 'farmspot_quality_4crop_manifest.json').read_text())
expected = set(manifest['classes'])
for split in ('train', 'val', 'test'):
    folders = {p.name for p in (dataset / split).iterdir() if p.is_dir()}
    assert folders == expected, (split, folders, expected)
    print(split, {name: len(list((dataset / split / name).glob('*.jpg'))) for name in sorted(folders)})
assert weights.is_file(), 'The pretrained tomato checkpoint is missing.'
assert torch.cuda.is_available(), 'Enable a GPU accelerator in Kaggle settings.'
print(torch.cuda.get_device_name(0))

In [ ]:
EPOCHS = 60
BATCH_SIZE = 32
IMAGE_SIZE = 224
SEED = 42

classifier = YOLO(str(weights), task='classify')
classifier.train(
    data=str(dataset), epochs=EPOCHS, imgsz=IMAGE_SIZE, batch=BATCH_SIZE,
    device=0, workers=2, seed=SEED, patience=15,
    project=str(working / 'farmspot_runs'), name='quality_classifier_4crop',
    auto_augment=None, erasing=0.0, hsv_h=0.0, hsv_s=0.0,
    hsv_v=0.0, fliplr=0.5, amp=False,
)
best_path = Path(classifier.trainer.best)
assert best_path.is_file(), best_path

In [ ]:
import shutil

from sklearn.metrics import classification_report, confusion_matrix

best = YOLO(str(best_path), task='classify')
test_images = sorted((dataset / 'test').glob('*/*.jpg'))
names = sorted(expected)
actual = [photo.parent.name for photo in test_images]
predicted = []
for start in range(0, len(test_images), BATCH_SIZE):
    batch = test_images[start:start + BATCH_SIZE]
    predictions = best.predict(
        [str(photo) for photo in batch], imgsz=IMAGE_SIZE,
        device=0, verbose=False,
    )
    predicted.extend(result.names[result.probs.top1] for result in predictions)
print(classification_report(actual, predicted, labels=names, zero_division=0))
report = classification_report(
    actual, predicted, labels=names, zero_division=0, output_dict=True,
)
(working / 'farmspot_quality_4crop_test_report.json').write_text(json.dumps(report, indent=2))
matrix = confusion_matrix(actual, predicted, labels=names).tolist()
(working / 'farmspot_quality_4crop_confusion.json').write_text(
    json.dumps({'labels': names, 'rows_actual_columns_predicted': matrix}, indent=2)
)
shutil.copy2(best_path, working / 'farmspot_quality_4crop_best.pt')